# 1. Carregando Banco de Dados 

In [ ]:
import pandas as pd

df = pd.read_csv("data/ecommerce_pedidos.csv")

print(df.shape)
print(df.isnull().sum())


# 2. Verificando dados nulos

In [ ]:
entregue = df["status"] == "Entregue"
nulo = df["data_entrega"].isnull()

entrega_sem_data = df[(entregue) & (nulo)]
print(len(entrega_sem_data)) 

# Existem 12 dados nulos na data de entregue com o pedido ja entregue.



# 3. Verificando Duplicatas

In [ ]:
cols = ["cliente", "produto", "quantidade", "valor_total", "data_pedido"]
duplicadas = df.duplicated(subset=cols)
print(duplicadas.sum())

df_limpo = df.drop_duplicates(subset=cols)

print("Antes:", df.shape)
print("Depois:", df_limpo.shape)

# 4. Verificando Datas Inválidas

In [ ]:
data = (~df["data_pedido"].str.startswith("2024")).sum()
print(data)

df_limpo["data_pedido"] = pd.to_datetime(df_limpo["data_pedido"], format="mixed", dayfirst=True)

print(df_limpo["data_pedido"].head(10))

tem_data = df_limpo["data_entrega"].notna()
data_invalida = (tem_data) & (df_limpo["data_entrega"] < df_limpo["data_pedido"])
df_limpo = df_limpo[~data_invalida]
print(df_limpo.shape)


# 5. Verificando Valores

In [54]:
valor_errado = df["valor_total"] <= 0 

print(valor_errado.sum())

df_limpo = df_limpo[df_limpo["valor_total"] > 0]
print(df_limpo.shape)




8
(442, 12)


# 6. Inconsistência de Cálculo

In [ ]:
df_limpo["valor_calculado"] = round(df_limpo["quantidade"] * df_limpo["valor_unitario"],2)

errado = df_limpo["valor_calculado"] != df_limpo["valor_total"]
print(errado.sum())

df_limpo.loc[errado, "valor_total"] = df_limpo.loc[errado, "valor_calculado"]
df_limpo = df_limpo.drop(columns=["valor_calculado"])


# 7. Salvando Dataset LIMPO

In [53]:
df_limpo.to_csv("data/ecommerce_pedidos_limpo.csv", index=False)
print("Arquivo salvo!")

Arquivo salvo!


## Resumo da Limpeza

Problema | Encontrado | Ação |
|---|---|---|
| Dados nulos (cliente, estado, pagamento) | 25 células | Identificado e documentado |
| Pedidos "Entregue" sem data | 12 linhas | Identificado e documentado |
| Duplicatas | 9 linhas | Removidas |
| Datas com formato errado | 15 linhas | Corrigidas para datetime |
| Valores impossíveis (zero/negativo) | 8 linhas | Removidas |
| valor_total incorreto | 20 linhas | Corrigidas com qtd × valor_unitario |
| Data entrega anterior ao pedido | 51 linhas | Removidas |

Dataset original: 510 linhas  
Dataset limpo: 442 linhas  
Registros removidos: 68 linhas